In [1]:
import boto3
import sagemaker
import torch

from sagemaker.estimator import Estimator
from sagemaker.model import Model

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sess = sagemaker.Session()
region = sess.boto_region_name

s3_client = boto3.client("s3", region_name=region)

sagemaker_role = sagemaker.get_execution_role()

bucket = sess.default_bucket()
bucket_prefix = "vision-demo"

bucket, bucket_prefix, sagemaker_role, region

('sagemaker-ap-southeast-2-879381285437',
 'vision-demo',
 'arn:aws:iam::879381285437:role/service-role/AmazonSageMaker-ExecutionRole-20241108T211736',
 'ap-southeast-2')

In [3]:
base = f"s3://{bucket}/{bucket_prefix}"

data_path = f"{base}/data"
model_dir = f"{base}/model"

x_train_file = "x_train.pt"
y_train_file = "y_train.pt"
x_test_file = "x_test.pt"
y_test_file = "y_test.pt"

In [9]:
num_classes = 5
boxes_per_cell = 3
grid_size = 7
hidden_size = 2048
vit_model_name = "google/vit-base-patch16-224-in21k"
learning_rate = 1e-5
batch_size = 32
epochs = 16

train_data_x = f"{data_path}/{x_train_file}"
train_data_y = f"{data_path}/{y_train_file}"
test_data_x = f"{data_path}/{x_test_file}"
test_data_y = f"{data_path}/{y_test_file}"

model_dir = "/opt/ml/model"

In [8]:
image_uri = sagemaker.image_uris.retrieve(framework="huggingface", region=region, version="4.4.2", image_scope="training", base_framework_version="pytorch1.6.0")

estimator = Estimator(
    image_uri=image_uri,
    role=sagemaker_role,
    instance_count=1,
    instance_type="ml.p3.2xlarge",
    output_path=model_dir,
    base_job_name="model-training",
    entry_point="train.py",
    source_dir=".",
    dependencies=["requirements.txt"]
)

estimator.set_hyperparameters(
    num_classes=num_classes,
    boxes_per_cell=boxes_per_cell,
    grid_size=grid_size,
    hidden_size=hidden_size,
    vit_model_name=vit_model_name,
    learning_rate=learning_rate,
    batch_size=batch_size,
    epochs=epochs,
    train_data_x=train_data_x,
    train_data_y=train_data_y,
    test_data_x=test_data_x,
    test_data_y=test_data_y,
    model_dir=model_dir
)

estimator.fit()